# curious-george — Phase 0 validation notebook

Runs the already-validated foundation (memory store, fabricated ground truth, loss measurement, LoRA fine-tuning) against GPU compute, then moves on to Phase 1 experiments. See the repo [README](https://github.com/stvenmobile/curious-george#readme) for the full roadmap and the definitions behind each term used below.

In [ ]:
%cd /content
!rm -rf /content/curious-george
!git clone https://github.com/stvenmobile/curious-george.git /content/curious-george
%cd /content/curious-george
!pip install -q -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, "/content/curious-george/src")

from curious_george.warble_harness import run_sanity_check
run_sanity_check(device="cuda")

## Persisting memory across sessions

Colab's local disk is wiped on every runtime restart. `MemoryStore`'s whole reason for existing is a `loss_history` that survives across sessions — without that, `learning_progress()` never has more than one measurement to compare against, and the "learn and retain" premise of the project doesn't hold.

This cell mounts Google Drive and points `MemoryStore.save()`/`load()` at a folder there via the `CURIOUS_GEORGE_MEMORY_DIR` environment variable, instead of the ephemeral local `data/memory/` default. Run it once per session, before any `save()`/`load()` call with no explicit path — it's read at call time, so it doesn't need to happen in the same cell as those calls.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["CURIOUS_GEORGE_MEMORY_DIR"] = "/content/drive/MyDrive/curious-george-memory"

from curious_george.memory_store import MemoryStore
store = MemoryStore.load()   # picks up last session's state from Drive, or starts empty on first run
print(f"Loaded {len(store)} item(s) from persistent memory.")

## The genuine learning test: LoRA fine-tuning, cold measurement

The sanity check above proved the plumbing works, but its "study" step just placed warble facts directly in the prompt — a confound, since *any* coherent context measurably helps next-token prediction (which is exactly why quaddles, whose own facts were never shown, still improved: +1.15 nats of that improvement is just reading-comprehension priming, not learning).

`run_lora_check` fixes this. It fine-tunes a small LoRA adapter (the base model's own weights are frozen and never change) on warble facts only, then measures loss on **cold** prompts — no warble content anywhere in the prompt, before or after fine-tuning. Any improvement can only come from the adapter's trained weights, not from priming. The original CPU run (`piper_assistant`, pre-port) took ~41 minutes and found: warbles improved 2.65 nats cold, quaddles (never trained on, structurally similar) leaked 0.68 nats (a ~26% residual confound from shared sentence structure between the two fabricated species — see the README's Phase 4). This run should reproduce that on GPU, much faster.

In [ ]:
from curious_george.warble_harness import run_lora_check
run_lora_check(device="cuda")

### Result (2026-09-12, T4 GPU)

| | before | after | delta | accuracy |
|---|---|---|---|---|
| warbles (LoRA-trained) | 3.4684 | 0.7177 | **+2.7507** | 0.483 -> 0.913 |
| quaddles (never trained on) | 3.6189 | 3.0267 | **+0.5922** | 0.271 -> 0.435 |

Leak ratio 0.215 — reproduces the original CPU run's finding (2.65 / 0.68 nats, ratio 0.258) almost exactly, on different hardware, with accuracy landing on the identical 0.913 / 0.435 both times. Confirms the port is sound and the result isn't hardware-specific.

Same interpretation as before: genuine, persistent, cold, mostly fact-specific learning from the LoRA adapter's own weights — real evidence for the core mechanism — with a real, if smaller, residual leak from warbles and quaddles sharing sentence structure. That leak is the open question Phase 4 in the README exists to address; Phase 1 (defining and testing a curiosity score) is the more immediate next step.

## Phase 1: does baseline loss category predict actual learning progress?

Phase 0's `recall_probes` were always decompositions of the *exact* sentences trained on — e.g. `study_facts` has "Warbles are green and yellow mammals." and the matching probe is that same sentence split into a prompt and target. That measures memorization: does the model recall the specific thing it was shown. It has never measured **generalization**: does studying some facts about a topic make the model any better at completing *different* material about that same topic.

That distinction is the actual definition of "noise" this project is using (see the README): a topic has exploitable structure if studying it transfers to unseen material; it's noise if the model can only memorize exactly what it saw, with nothing carrying over.

**This section went through two failed designs before landing on the one below — both failures are real findings, not just bugs, and are recorded in "What didn't work" further down rather than edited out.** The short version: generalization can't be measured by withholding some of a topic's *own* facts from training, because a small-capacity LoRA adapter trained on several completions of the same prefix (e.g. "Warbles...") builds a near-deterministic mapping from that prefix to those specific completions — a withheld fact sharing the same prefix collides with that mapping and gets *actively worse*, regardless of how much real structure the topic has. That's a genuine phenomenon (interference), just a different one than what this section is trying to measure.

The design that actually isolates generalization: pair each topic with a **separate sibling topic** — same template/register, different subject and vocabulary, never trained on at all. This is exactly how Phase 0 already measured the warble→quaddle leak, just formalized across three anchor categories:

- **known** — real-world common-knowledge sentences, paired with a second, unrelated set of common-knowledge sentences.
- **moderate** — Phase 0's original 10 warble facts, paired with quaddles (the same pair Phase 0 already validated).
- **noise** — word-salad sentences, paired with a second word-salad set drawn from a completely disjoint 64-word vocabulary (verified programmatically — zero shared words with the trained set).

`run_topic_trial` measures loss on both the topic's own `trained_probes` (memorization) and its sibling's probes (generalization) before and after a LoRA study step (200 steps, 1e-4 — Phase 0's validated regime) on `train_facts` only. `generalization_progress / memorization_progress` is the noise measurement: near 0 means nothing transferred to the sibling; positive means it did.

A first single run (one trial per topic) came back matching the hypothesis's predicted ordering exactly - but a single run per topic is one draw from a noisy training process, not a result worth trusting on its own. `run_noise_experiment_repeated` runs 5 independent trials per topic (a different LoRA training seed each time, so the fact-sampling order genuinely varies) and reports mean +/- stdev, so the numbers below reflect run-to-run variance rather than one possibly-lucky (or unlucky) draw.

In [ ]:
from curious_george.curiosity import run_noise_experiment_repeated
results = run_noise_experiment_repeated(device="cuda", num_trials=5)

### Result (2026-09-13/14, T4 GPU, 5 trials per topic)

| topic | n | mem mean | mem stdev | gen mean | gen stdev | ratio mean | ratio stdev |
|---|---|---|---|---|---|---|---|
| known | 5 | 0.6413 | 0.0004 | -0.4017 | 0.2898 | -0.626 | 0.452 |
| moderate | 5 | 2.6379 | 0.2030 | **+0.6389** | 0.1732 | **+0.246** | 0.076 |
| noise | 5 | 8.8902 | 0.1309 | **-1.8394** | 0.2873 | **-0.207** | 0.032 |

The strong parts of the hypothesis hold up robustly under replication: `moderate` is positive in all 5 trials (0.36 to 0.83 nats) - genuine, repeatable transfer. `noise` is negative in all 5 trials, and *tightly* so (ratio stdev only 0.032, the tightest of the three) - a highly consistent, severe penalty every time, not one unlucky draw.

One part of the original hypothesis needs revising, though. `known` was predicted to sit near zero (ceiling effect - already known, little room to move). Instead it's negative in all 5 trials too - smaller in raw magnitude than noise (-0.40 vs -1.84 nats) but the same sign, not neutral.

The real three-way picture looks less like "positive / neutral / very negative" and more like: **training produces net-positive transfer only when the content has genuine template structure to exploit (moderate); everything else - whether disconnected real facts or pure word salad - shows net-negative transfer, with severity scaling with how little structure is actually there.** Known facts don't share a reusable template with each other any more than noise sentences do; they just start from a much lower baseline loss, so the damage is smaller in absolute terms, not different in kind.

`known`'s large ratio stdev (0.452 against a mean of -0.626) is mostly an artifact of dividing by a tiny, extremely stable denominator (memorization stdev only 0.0004 - these facts get memorized almost identically every trial). The raw `generalization_progress` numbers are the more trustworthy comparison across categories when memorization magnitudes differ this much, and those are unambiguous: known's generalization spread (-0.09 to -0.75) never crosses into positive territory either.

**Where this leaves the curiosity-score idea**: the README's original sketch assumed a "known" anchor near zero, with moderate and noise on either side. This result says the real dividing line isn't baseline loss alone - it's whether the topic has genuine template structure to transfer, which a topic's baseline loss doesn't tell you on its own (known's baseline loss is very low, noise's very high, but both show net-negative generalization; only moderate, in the middle, transfers positively). Any curiosity_score built from this needs to predict *transferability*, not just target a baseline-loss sweet spot.

### What didn't work (kept for the record, not edited out)

**Attempt 1 — held-out facts, shared vocabulary noise content.** `noise`'s word-salad sentences reused ~25 words across all 10 sentences. Result: `noise`'s generalization ratio (0.420) came out *higher* than `moderate`'s (0.121) — backwards from the hypothesis. Cause: training on the trained noise sentences raised the model's probability on words the held-out sentences happened to share, which is real loss improvement but from vocabulary overlap, not structure.

**Attempt 2 — held-out facts, vocabulary leak fixed (80 distinct words, zero reuse).** All three categories came back *negative* this time — including `moderate`:

| topic | memorization | generalization | ratio |
|---|---|---|---|
| known | 0.6243 | -0.3197 | -0.512 |
| moderate | 2.7042 | -0.4144 | -0.153 |
| noise | 6.5294 | -0.4635 | -0.071 |

Fixing the noise leak didn't fix the real problem — it just exposed the next one. A controlled follow-up at Phase 0's own validated hyperparameters (200 steps, 1e-4, ruling out "too aggressive a cheap-trial learning rate" as the cause) made it *worse*, not better: `moderate`'s held-out generalization dropped to **-2.2741** (loss went from 3.65 to 5.92 — much worse than before training). Meanwhile the exact same training run improved Phase 0's *quaddle* probes by **+0.7134** — nearly matching Phase 0's original +0.68.

Same model, same training run, opposite signs, depending only on whether the probe shares the trained prefix. Mechanism: training on 8 different completions of `"Warbles..."` teaches the adapter a narrow, near-deterministic mapping from that prefix to those specific completions. A held-out fact sharing the same prefix collides with that mapping and gets actively worse — that's interference, not a measurement of whether the topic has learnable structure. A probe with a different subject (quaddles) never triggers the collision, so the positive number there is the genuine structural leak Phase 0 already found. Phase 0 never surfaced this because `run_lora_check` always trained on the *full* set of 10 warble facts together — nothing sharing that prefix was ever held back to collide with.

This is why the design above uses a separate sibling topic instead of same-topic held-out facts.

## Phase 2: the interest pipeline - cheap entry, expensive deep-scoring

Phase 1 answered "does this topic have real structure" for three hand-built anchor topics. The next question is architectural: given a stream of real, arbitrary candidate topics (not fixture data), how does one become part of Piper's memory, get ranked, and either get pursued, archived, or pruned? Full design discussion in `obsidian/Journals/2026-09-13.md` and `2026-09-14.md` - the short version:

- **One store, not two.** `MemoryStore`'s `MemoryItem` now carries `status` (`candidate` / `active` / `archived`) and a `deep_score_history` alongside the existing `loss_history` - a merely-noticed topic sits in the same store as a thoroughly-studied one, just earlier on the same axes.
- **Two-tier cost model.** Getting a candidate INTO the store (`pipeline.add_candidate`) only needs a cheap baseline loss reading (`measure_content_loss` - a plain forward pass, no training, no hand-authored probes) plus an embedding. The expensive step - a real LoRA trial, the thing that actually distinguishes learnable structure from noise - is reserved for deep-scoring, run only against candidates actually being considered.
- **Resonance**, alongside mastery, is the third axis: cosine similarity between a candidate's embedding and `my_interests.json` (five declared research interests). Verified for real below in the sanity-check cell - not a fixture test, actual MiniLM embeddings.
- **The sibling problem.** Phase 1's transferability measurement needs a domain-matched sibling (warbles/quaddles). Real candidates don't come with one, and building a bespoke sibling per candidate is impractical. Solution: three fixed "canary" topics from genuinely offbeat, unrelated domains (lighthouse keeping, vintage typewriters, traditional knots - see `canary_topics.json`) stand in for a matched sibling. This measures something related but different from Phase 1's original signal: not "did this candidate's specific structure transfer to a matched partner," but "did fine-tuning on this candidate produce a well-behaved update, or a narrow, corrupting overfit" - a generalization-quality canary, not a domain-matched structural-transfer probe.
- **Deep-scoring the pipeline** (`deep_scoring.run_deep_scoring_pass`) 5-trial-averages the canary-based transferability check against every `candidate`-status item, then prunes anything below threshold (or unmeasurable) and promotes everything else to `active`. Deliberately doesn't archive anything - archiving (mastered, no new sources available) is a judgment this pass can't make on its own.

The cell below runs the real, non-fixture sanity check for the cheap-entry half: three genuine candidate topics (one thematically aligned with a declared interest, two unrelated), through the real model and real MiniLM embedder.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from curious_george.pipeline import add_candidates
from curious_george.embeddings import SmallModelEmbedder
from curious_george.my_interests import load_my_interests, embed_my_interests
from curious_george.memory_store import MemoryStore

pipeline_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
pipeline_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct").to("cuda")
pipeline_model.eval()
pipeline_embedder = SmallModelEmbedder()

interests = load_my_interests()
interest_embeddings = embed_my_interests(interests, pipeline_embedder)

candidates = [
    {"topic": "goal_arbitration", "content": "How should an autonomous agent decide which of several competing goals to pursue right now."},
    {"topic": "sourdough_starters", "content": "A sourdough starter is a fermented mixture of flour and water used to leaven bread."},
    {"topic": "the_paris_agreement", "content": "The Paris Agreement is an international treaty on climate change adopted in 2015."},
]

interest_pipeline_store = MemoryStore()
cheap_scores = add_candidates(interest_pipeline_store, pipeline_model, pipeline_tokenizer, "cuda",
                               pipeline_embedder, candidates, interest_embeddings)
for score in cheap_scores:
    print(score)

### Result (2026-09-14, local CPU run - real model, real MiniLM)

| topic | mastery | resonance |
|---|---|---|
| goal_arbitration (agent goal-selection) | novice | **0.76** |
| sourdough_starters (unrelated) | novice | 0.05 |
| the_paris_agreement (unrelated) | novice | 0.06 |

Real discrimination, not a toy result: `goal_arbitration` - genuinely about the declared "goal prioritization"/"agentic behavior" interests - scores dramatically higher resonance than two topically unrelated candidates. All three land as "novice" mastery (baseline losses 1.79-4.21), which is reasonable - none are gibberish, none are already perfectly known.

### Deep-scoring these same candidates

Cheap scoring only tells us mastery and resonance - not whether studying any of these would actually be worthwhile (transfer well) or actively harmful (the noise-like interference Phase 1 found). `run_deep_scoring_pass` runs the 5-trial-averaged canary check against every `candidate`-status item in the store, then prunes anything below threshold and promotes the rest to `active`.

In [ ]:
from curious_george.deep_scoring import run_deep_scoring_pass

deep_score_results = run_deep_scoring_pass(interest_pipeline_store, "Qwen/Qwen2.5-0.5B-Instruct", "cuda")
for topic, outcome in deep_score_results.items():
    print(topic, outcome["outcome"], outcome["record"])